In [6]:
import requests
import pandas as pd

In [7]:
#LA Weather data from Open-Meteo


url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 34.05,
    "longitude": -118.24,
    "start_date": "2022-01-01",
    "end_date": "2025-01-01",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature"
    ],
    "timezone": "America/Los_Angeles"
}

response = requests.get(url, params=params)
data = response.json()

In [8]:
weather = pd.DataFrame({
    "time": data["hourly"]["time"],
    "temperature": data["hourly"]["temperature_2m"],
    "humidity": data["hourly"]["relative_humidity_2m"],
    "apparent_temperature": data["hourly"]["apparent_temperature"]
})

In [9]:
weather["datetime"] = pd.to_datetime(weather["time"])
weather = weather.drop(columns=["time"])

weather = weather.sort_values("datetime")
weather["datetime"] = weather["datetime"].dt.floor("H")

weather["hour"] = weather["datetime"].dt.hour
weather["month"] = weather["datetime"].dt.month

In [ ]:
#EIA CAISO Demand Data 

API_KEY = "YOUR_API_KEY"

url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

all_records = []
offset = 0

while True: 
    params = {
        "api_key": API_KEY,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "CISO", 
        "start": "2022-01-01",
        "end": "2025-01-01",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": offset,
        "length": 5000
    }

    #Fetch Data 
    response = requests.get(url, params=params)
    data = response.json()
    records = data["response"]["data"]
    if not records:
        break
    all_records.extend(records)
    if len(records) < 5000:
        break
    offset += 5000

data = {"response": {"data": all_records}}



In [11]:
#Covert to DataFrame 
records = data["response"]["data"]

eia_df = pd.DataFrame(records)

eia_df = eia_df[eia_df["type"] == "D"]  # keep only real demand
eia_df.head()

,period,respondent,respondent-name,type,type-name,value,value-units
0,2022-01-01T00,CISO,California Independent System Operator,D,Demand,22618,megawatthours
4,2022-01-01T01,CISO,California Independent System Operator,D,Demand,24545,megawatthours
8,2022-01-01T02,CISO,California Independent System Operator,D,Demand,27081,megawatthours
12,2022-01-01T03,CISO,California Independent System Operator,D,Demand,27153,megawatthours
16,2022-01-01T04,CISO,California Independent System Operator,D,Demand,26522,megawatthours


In [12]:
#Clean Up Data 
eia_df = eia_df[["period", "value"]]

eia_df = eia_df.rename(columns={
    "period": "datetime",
    "value": "demand_mwh" #megawatt-hours (MWh) (electricity demand measured in energy used over time)
})
eia_df.head()

,datetime,demand_mwh
0,2022-01-01T00,22618
4,2022-01-01T01,24545
8,2022-01-01T02,27081
12,2022-01-01T03,27153
16,2022-01-01T04,26522


In [13]:
weather["datetime"] = pd.to_datetime(weather["datetime"])
eia_df["datetime"] = pd.to_datetime(eia_df["datetime"])

merged_df = pd.merge(
    weather,
    eia_df,
    on="datetime",
    how="inner"
)

merged_df.head()

,temperature,humidity,apparent_temperature,datetime,hour,month,demand_mwh
0,8.1,77,5.0,2022-01-01 00:00:00,0,1,22618
1,6.3,79,3.4,2022-01-01 01:00:00,1,1,24545
2,5.2,77,2.4,2022-01-01 02:00:00,2,1,27081
3,5.0,76,2.1,2022-01-01 03:00:00,3,1,27153
4,3.7,80,0.8,2022-01-01 04:00:00,4,1,26522


In [14]:
#Save data to csv 
weather.to_csv("weather.csv", index=False)
eia_df.to_csv("eia.csv", index=False)